# Prepare food-viz tables


## Import libraries and define paths


In [1]:
from pathlib import Path

import pandas as pd

# Notebook lives in data/processed/ — run with that as cwd (or open from there in Jupyter).
DATA = Path(".").resolve()
FINAL = (DATA / "../final").resolve()
FINAL.mkdir(parents=True, exist_ok=True)

# Inputs + ER audit stay in processed/; deliverable tables go to ../final/
PATHS = {
    "establishment_keys": DATA / "establishment_keys.csv",
    "merged_inspections": DATA / "merged_food_inspections.csv",
    "category": DATA / "category-Utah_food.csv",
    "meta": DATA / "meta-Utah_food.csv",
    "review": DATA / "review-Utah_food.csv",
    # intermediate (processed/)
    "match_audit": DATA / "meta_slchd_match_audit.csv",
    # outputs → data/final/
    "establishment_keys_geocoded": FINAL / "establishment_keys_geocoded.csv",
    "establishment_keys_final": FINAL / "establishment_keys.csv",
    "meta_matched": FINAL / "meta-Utah_food_matched.csv",
    "establishments": FINAL / "establishments.csv",
    "establishment_attributes": FINAL / "establishment_attributes.csv",
    "establishment_categories": FINAL / "establishment_categories.csv",
    "gmap_hours": FINAL / "gmap_hours.csv",
    "merged_inspections_clean": FINAL / "merged_food_inspections.csv",
    "inspection_types": FINAL / "inspection_types.csv",
    "inspections": FINAL / "inspections.csv",
    "violations": FINAL / "violations.csv",
    "violation_codes": FINAL / "violation_codes.csv",
}

pd.DataFrame(
    [
        {
            "key": k,
            "file": p.name,
            "dir": p.parent.name,
            "size_mb": round(p.stat().st_size / 1e6, 1) if p.is_file() else None,
            "exists": p.is_file(),
        }
        for k, p in PATHS.items()
    ]
)


,key,file,dir,size_mb,exists
0,establishment_keys,establishment_keys.csv,processed,0.2,True
1,merged_inspections,merged_food_inspections.csv,processed,10.1,True
2,category,category-Utah_food.csv,processed,0.0,True
3,meta,meta-Utah_food.csv,processed,8.2,True
4,review,review-Utah_food.csv,processed,765.5,True
5,match_audit,meta_slchd_match_audit.csv,processed,0.0,True
6,establishment_keys_geocoded,establishment_keys_geocoded.csv,final,0.1,True
7,establishment_keys_final,establishment_keys.csv,final,0.3,True
8,meta_matched,meta-Utah_food_matched.csv,final,8.2,True
9,establishments,establishments.csv,final,1.7,True


In [2]:
establishment_keys = pd.read_csv(PATHS["establishment_keys"])
# low_memory=False: mixed null/string columns (rank, violation_*) otherwise warn on chunked dtype inference
merged_inspections = pd.read_csv(PATHS["merged_inspections"], low_memory=False)
category = pd.read_csv(PATHS["category"])
meta = pd.read_csv(PATHS["meta"])
# 17 gmap_ids are exact full-row duplicates in the source — drop before any
# downstream table keys on gmap_id (establishments, gmap_hours, etc.)
meta = meta.drop_duplicates(subset="gmap_id").reset_index(drop=True)
review = pd.read_csv(PATHS["review"])

{
    "establishment_keys": establishment_keys.shape,
    "merged_inspections": merged_inspections.shape,
    "category": category.shape,
    "meta": meta.shape,
    "review": review.shape,
}


{'establishment_keys': (1341, 7),
 'merged_inspections': (33169, 19),
 'category': (685, 2),
 'meta': (8592, 25),
 'review': (3602447, 7)}

## Fix inconsistent violation criticality

Three `violation_code` values in `merged_inspections` are marked both
critical and non-critical across rows, with one side a small minority
against the other (see investigation): `4.4.23**` (food thermometer
accuracy), `4.6.24**` (no soap at hand sink), `4.6.25**` (no hand-drying
provision at hand sink). All three are genuinely critical food-safety
issues, so normalize `violation_critical` to `True` for every row with
these codes rather than keep the source's inconsistent labeling.


In [3]:
CRITICAL_CODE_FIXES = ["4.4.23**", "4.6.24**", "4.6.25**"]

fix_mask = merged_inspections["violation_code"].isin(CRITICAL_CODE_FIXES)
before = merged_inspections.loc[fix_mask].groupby(
    ["violation_code", "violation_critical"]
).size()

merged_inspections.loc[fix_mask, "violation_critical"] = True

after = merged_inspections.loc[fix_mask].groupby(
    ["violation_code", "violation_critical"]
).size()

merged_inspections.to_csv(PATHS["merged_inspections_clean"], index=False)

{"before": before.to_dict(), "after": after.to_dict()}


{'before': {('4.4.23**', False): 2,
  ('4.4.23**', True): 2,
  ('4.6.24**', False): 3,
  ('4.6.24**', True): 196,
  ('4.6.25**', False): 3,
  ('4.6.25**', True): 421},
 'after': {('4.4.23**', True): 4,
  ('4.6.24**', True): 199,
  ('4.6.25**', True): 424}}

## Entity resolution: `meta` ↔ `establishment_keys`

Match Google Local (`meta`) places to SLCHD establishments by:

1. **Name + address similarity** (normalized string ratio + token Jaccard)
2. **Geographic confirmation** — Census-geocode SLCHD street addresses, then require the meta `(lat, lon)` to fall within ~150 m (same location on the map)
3. **Text-only fallback** — if Census geocoding fails for an SLCHD row, still match when name *and* address are extremely similar (near-exact), blocked by street number + city

Each meta row keeps at most one best SLCHD key (`SLCHD_establishment_keys`). Geo-confirmed matches win over text-only when both exist.


### Normalize strings + geocode SLCHD addresses


In [4]:
import csv
import math
import re
from difflib import SequenceMatcher

import numpy as np
import plotly.express as px
import requests

# Expand common USPS-style tokens so "S STATE ST" and "South State Street" align.
_DIR = {
    "N": "NORTH", "S": "SOUTH", "E": "EAST", "W": "WEST",
    "NE": "NORTHEAST", "NW": "NORTHWEST", "SE": "SOUTHEAST", "SW": "SOUTHWEST",
}
_SUF = {
    "ST": "STREET", "AVE": "AVENUE", "AV": "AVENUE", "BLVD": "BOULEVARD",
    "DR": "DRIVE", "RD": "ROAD", "LN": "LANE", "CT": "COURT", "CIR": "CIRCLE",
    "PL": "PLACE", "PKWY": "PARKWAY", "HWY": "HIGHWAY", "WAY": "WAY", "TER": "TERRACE",
}


def norm_tokens(s: str) -> str:
    s = re.sub(r"[^A-Z0-9 ]+", " ", str(s).upper())
    return " ".join(_DIR.get(t, _SUF.get(t, t)) for t in s.split())


def norm_name(s: str) -> str:
    """Normalize trade names; strip legal/boilerplate but keep brand-distinct words (Coffee/Cafe/Bar)."""
    s = norm_tokens(s)
    # SLCHD sometimes suffixes outdoor seating as a separate permit line
    s = re.sub(r"\bDOG PATIO\b", " ", s)
    for junk in ("THE", "RESTAURANT", "LLC", "INC", "AND"):
        s = re.sub(rf"\b{junk}\b", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def street_from_meta_address(addr) -> str:
    """meta address is usually 'Name, STREET, City, ST ZIP' — take the street field."""
    if pd.isna(addr):
        return ""
    parts = [p.strip() for p in str(addr).split(",")]
    if len(parts) >= 2 and not re.match(r"^\d", parts[0]):
        return norm_tokens(parts[1])
    return norm_tokens(parts[0])


def city_from_meta_address(addr) -> str:
    """City field from meta address ('Name, STREET, City, ST ZIP')."""
    if pd.isna(addr):
        return ""
    parts = [p.strip() for p in str(addr).split(",")]
    return norm_tokens(parts[-2]) if len(parts) >= 3 else ""


def string_sim(a: str, b: str) -> float:
    if not a or not b:
        return 0.0
    return SequenceMatcher(None, a, b).ratio()


def token_jaccard(a: str, b: str) -> float:
    ta, tb = set(a.split()), set(b.split())
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)


def haversine_m(lat1, lon1, lat2, lon2) -> float:
    """Great-circle distance in meters (spherical Earth)."""
    r = 6_371_000.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


def geocode_establishment_keys(ek: pd.DataFrame, cache_path: Path) -> pd.DataFrame:
    """Attach Census geocoder lat/lon to SLCHD rows (cached — portal rows have no coordinates)."""
    if cache_path.is_file():
        cached = pd.read_csv(cache_path)
        return ek.merge(cached, on="establishment_key", how="left")

    rows = []
    for i, r in ek.reset_index(drop=True).iterrows():
        parts = [p.strip() for p in str(r["city_state_zip"]).split(",")]
        city = parts[0] if parts else ""
        rest = parts[1] if len(parts) > 1 else ""
        toks = rest.split()
        state = toks[0] if toks else "UT"
        zipc = toks[1] if len(toks) > 1 else ""
        # Census batch format: id, street, city, state, zip
        rows.append(f"{i},{r['address']},{city},{state},{zipc}")

    resp = requests.post(
        "https://geocoding.geo.census.gov/geocoder/locations/addressbatch",
        files={"addressFile": ("addrs.csv", ("\n".join(rows)).encode())},
        data={"benchmark": "Public_AR_Current"},
        timeout=300,
    )
    resp.raise_for_status()

    geo_rows = []
    ek_reset = ek.reset_index(drop=True)
    for line in resp.text.strip().splitlines():
        cols = next(csv.reader([line]))
        rid = int(cols[0])
        status = cols[2]
        lat = lon = np.nan
        if status == "Match" and len(cols) > 5 and cols[5]:
            lon_s, lat_s = cols[5].split(",")
            lon, lat = float(lon_s), float(lat_s)
        geo_rows.append(
            {
                "establishment_key": ek_reset.loc[rid, "establishment_key"],
                "slchd_lat": lat,
                "slchd_lon": lon,
                "geocode_status": status,
            }
        )

    geo = pd.DataFrame(geo_rows)
    geo.to_csv(cache_path, index=False)
    return ek.merge(geo, on="establishment_key", how="left")


ek_geo = geocode_establishment_keys(establishment_keys, PATHS["establishment_keys_geocoded"])
{
    "slchd_rows": len(ek_geo),
    "geocode_match": int(ek_geo["slchd_lat"].notna().sum()),
    "geocode_miss": int(ek_geo["slchd_lat"].isna().sum()),
}


{'slchd_rows': 1341, 'geocode_match': 1270, 'geocode_miss': 71}

### Fill remaining geocode misses via Nominatim

Census batch geocoding is strict about street/city/zip formatting and misses
~148 of 1,341 SLCHD addresses. Retry just those through Nominatim (OSM), which
parses a single free-form address string more tolerantly. Cached results are
merged back into `establishment_keys_geocoded.csv` so reruns don't re-query.


In [5]:
from geopy.extra.rate_limiter import RateLimiter
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="food-health-viz-research (arlethssbb@gmail.com)")
geocode_osm = RateLimiter(geolocator.geocode, min_delay_seconds=1.1)

miss_mask = ek_geo["slchd_lat"].isna()
n_before = int(miss_mask.sum())

for i in ek_geo[miss_mask].index:
    r = ek_geo.loc[i]
    loc = geocode_osm(f"{r['address']}, {r['city_state_zip']}")
    if loc is None:
        continue
    ek_geo.loc[i, "slchd_lat"] = loc.latitude
    ek_geo.loc[i, "slchd_lon"] = loc.longitude
    ek_geo.loc[i, "geocode_status"] = "Match_Nominatim"

# Cache file only carries the key + geocode columns, matching its original schema.
ek_geo[["establishment_key", "slchd_lat", "slchd_lon", "geocode_status"]].to_csv(
    PATHS["establishment_keys_geocoded"], index=False
)

n_after = int(ek_geo["slchd_lat"].isna().sum())
{"census_misses": n_before, "nominatim_recovered": n_before - n_after, "still_missing": n_after}


{'census_misses': 71, 'nominatim_recovered': 0, 'still_missing': 71}

### Write final establishment_keys table

Rename the key column to `slchd_establishment_key` (consistent naming used
across every new table below) and carry the geocoded lat/lon straight through
as columns — the DuckDB loader turns them into a `geometry` column.


In [6]:
establishment_keys_final = ek_geo.rename(
    columns={"establishment_key": "slchd_establishment_key"}
)
establishment_keys_final.to_csv(PATHS["establishment_keys_final"], index=False)

{
    "out": str(PATHS["establishment_keys_final"]),
    "rows": len(establishment_keys_final),
    "geocoded": int(establishment_keys_final["slchd_lat"].notna().sum()),
}


{'out': '/Users/arleth/Desktop/Research_Work/food-health-viz/data/final/establishment_keys.csv',
 'rows': 1341,
 'geocoded': 1270}

### Score candidates (geo-confirmed, then text fallback for geocode misses)


In [7]:
# Geo path: distance gate + moderate text gates.
# Text fallback: ungeocoded SLCHD only, near-exact name/address (co-tenant false positives are rare at this bar).
MAX_DIST_M = 150.0
MIN_NAME_SIM = 0.80
MIN_NAME_JAC = 0.55
MIN_ADDR_SIM = 0.60
MIN_SCORE = 0.78

# Stricter than geo path — no lat/lon safety net when Census fails.
TEXT_MIN_NAME_SIM = 0.92
TEXT_MIN_NAME_JAC = 0.80
TEXT_MIN_ADDR_SIM = 0.88
TEXT_MIN_CITY_SIM = 0.80
TEXT_MIN_SCORE = 0.90

ek_all = ek_geo.copy()
ek_all["name_n"] = ek_all["name"].map(norm_name)
ek_all["addr_n"] = ek_all["address"].map(norm_tokens)
ek_all["city_n"] = ek_all["city"].map(norm_tokens)
ek_all["stnum"] = ek_all["addr_n"].str.extract(r"^(\d+)", expand=False).fillna("")

ek_geo_ok = ek_all.dropna(subset=["slchd_lat", "slchd_lon"]).copy()
ek_geo_miss = ek_all[ek_all["slchd_lat"].isna()].copy()

meta_m = meta.copy()
meta_m["name_n"] = meta_m["name"].map(norm_name)
meta_m["addr_n"] = meta_m["address"].map(street_from_meta_address)
meta_m["city_n"] = meta_m["address"].map(city_from_meta_address)
meta_m["stnum"] = meta_m["addr_n"].str.extract(r"^(\d+)", expand=False).fillna("")
meta_m = meta_m.reset_index(drop=True)


def consider(best: dict, j: int, row: dict) -> None:
    prev = best.get(j)
    if prev is None or row["score"] > prev["score"]:
        best[j] = row


best: dict[int, dict] = {}

# --- Pass 1: geo-confirmed ---
for _, er in ek_geo_ok.iterrows():
    dlat = MAX_DIST_M / 111_000.0
    dlon = MAX_DIST_M / (111_000.0 * math.cos(math.radians(er["slchd_lat"])))
    near = meta_m[
        meta_m["latitude"].between(er["slchd_lat"] - dlat, er["slchd_lat"] + dlat)
        & meta_m["longitude"].between(er["slchd_lon"] - dlon, er["slchd_lon"] + dlon)
    ]
    for j, mr in near.iterrows():
        if pd.isna(mr["latitude"]) or pd.isna(mr["longitude"]):
            continue
        dist = haversine_m(er["slchd_lat"], er["slchd_lon"], mr["latitude"], mr["longitude"])
        if dist > MAX_DIST_M:
            continue
        name_sim = string_sim(er["name_n"], mr["name_n"])
        name_jac = token_jaccard(er["name_n"], mr["name_n"])
        addr_sim = string_sim(er["addr_n"], mr["addr_n"])
        if name_sim < MIN_NAME_SIM or name_jac < MIN_NAME_JAC or addr_sim < MIN_ADDR_SIM:
            continue
        score = (
            0.45 * name_sim
            + 0.25 * name_jac
            + 0.20 * addr_sim
            + 0.10 * (1.0 - dist / MAX_DIST_M)
        )
        if score < MIN_SCORE:
            continue
        consider(
            best,
            j,
            {
                "gmap_id": mr["gmap_id"],
                "SLCHD_establishment_keys": er["establishment_key"],
                "meta_name": mr["name"],
                "slchd_name": er["name"],
                "meta_address": mr["address"],
                "slchd_address": f"{er['address']}, {er['city_state_zip']}",
                "name_sim": name_sim,
                "name_jac": name_jac,
                "addr_sim": addr_sim,
                "dist_m": dist,
                "score": score,
                "match_method": "geo",
                "meta_lat": mr["latitude"],
                "meta_lon": mr["longitude"],
                "slchd_lat": er["slchd_lat"],
                "slchd_lon": er["slchd_lon"],
            },
        )

# --- Pass 2: text-only for Census misses (skip meta already geo-matched) ---
geo_matched_idx = set(best.keys())
for _, er in ek_geo_miss.iterrows():
    cand = meta_m
    if er["stnum"]:
        cand = cand[cand["stnum"] == er["stnum"]]
    if er["city_n"]:
        # Same city (or containment) — blocks cross-valley chain collisions.
        city = er["city_n"]
        cand = cand[
            cand["city_n"].eq(city)
            | cand["city_n"].str.contains(city, na=False)
            | cand["city_n"].map(lambda c: isinstance(c, str) and city in c)
        ]
    for j, mr in cand.iterrows():
        if j in geo_matched_idx:
            continue
        name_sim = string_sim(er["name_n"], mr["name_n"])
        name_jac = token_jaccard(er["name_n"], mr["name_n"])
        addr_sim = string_sim(er["addr_n"], mr["addr_n"])
        city_sim = string_sim(er["city_n"], mr["city_n"])
        if (
            name_sim < TEXT_MIN_NAME_SIM
            or name_jac < TEXT_MIN_NAME_JAC
            or addr_sim < TEXT_MIN_ADDR_SIM
            or city_sim < TEXT_MIN_CITY_SIM
        ):
            continue
        # No distance term — pure text agreement must be extreme.
        score = 0.50 * name_sim + 0.25 * name_jac + 0.25 * addr_sim
        if score < TEXT_MIN_SCORE:
            continue
        consider(
            best,
            j,
            {
                "gmap_id": mr["gmap_id"],
                "SLCHD_establishment_keys": er["establishment_key"],
                "meta_name": mr["name"],
                "slchd_name": er["name"],
                "meta_address": mr["address"],
                "slchd_address": f"{er['address']}, {er['city_state_zip']}",
                "name_sim": name_sim,
                "name_jac": name_jac,
                "addr_sim": addr_sim,
                "dist_m": float("nan"),
                "score": score,
                "match_method": "text_fallback",
                "meta_lat": mr["latitude"],
                "meta_lon": mr["longitude"],
                "slchd_lat": float("nan"),
                "slchd_lon": float("nan"),
            },
        )

match_audit = (
    pd.DataFrame(best.values()).sort_values("score", ascending=False).reset_index(drop=True)
    if best
    else pd.DataFrame(
        columns=[
            "gmap_id",
            "SLCHD_establishment_keys",
            "meta_name",
            "slchd_name",
            "name_sim",
            "name_jac",
            "addr_sim",
            "dist_m",
            "score",
            "match_method",
        ]
    )
)
match_audit.to_csv(PATHS["match_audit"], index=False)

{
    "matched_meta_rows": len(match_audit),
    "unique_slchd_keys": int(match_audit["SLCHD_establishment_keys"].nunique())
    if len(match_audit)
    else 0,
    "by_method": match_audit["match_method"].value_counts().to_dict()
    if len(match_audit)
    else {},
    "slchd_geocoded": len(ek_geo_ok),
    "slchd_geocode_miss": len(ek_geo_miss),
    "meta_rows": len(meta_m),
}


/var/folders/9g/_cb64y1n6wqdkyfbjx16g3f80000gn/T/ipykernel_29231/25200335.py:101: Pandas4Warning: 'or' operations between boolean dtype and str are deprecated and will raise in a future version. Explicitly cast the strings to a boolean dtype before operating instead.
  cand["city_n"].eq(city)
/var/folders/9g/_cb64y1n6wqdkyfbjx16g3f80000gn/T/ipykernel_29231/25200335.py:101: Pandas4Warning: 'or' operations between boolean dtype and str are deprecated and will raise in a future version. Explicitly cast the strings to a boolean dtype before operating instead.
  cand["city_n"].eq(city)
/var/folders/9g/_cb64y1n6wqdkyfbjx16g3f80000gn/T/ipykernel_29231/25200335.py:101: Pandas4Warning: 'or' operations between boolean dtype and str are deprecated and will raise in a future version. Explicitly cast the strings to a boolean dtype before operating instead.
  cand["city_n"].eq(city)
/var/folders/9g/_cb64y1n6wqdkyfbjx16g3f80000gn/T/ipykernel_29231/25200335.py:101: Pandas4Warning: 'or' operations betw

{'matched_meta_rows': 139,
 'unique_slchd_keys': 138,
 'by_method': {'geo': 139},
 'slchd_geocoded': 1270,
 'slchd_geocode_miss': 71,
 'meta_rows': 8592}

In [8]:
cols = [
    "meta_name",
    "slchd_name",
    "match_method",
    "name_sim",
    "name_jac",
    "addr_sim",
    "dist_m",
    "score",
]
match_audit[cols].head(15)


,meta_name,slchd_name,match_method,name_sim,name_jac,addr_sim,dist_m,score
0,Town & Country Market,TOWN & COUNTRY MARKET,geo,1.0,1.0,1.0,15.892878,0.989405
1,Navajo Hogan,NAVAJO HOGAN,geo,1.0,1.0,1.0,18.883076,0.987411
2,Stella Grill,STELLA GRILL,geo,1.0,1.0,1.0,19.987356,0.986675
3,Apollo Burger,APOLLO BURGER,geo,1.0,1.0,1.0,20.178363,0.986548
4,Chick-fil-A,CHICK FIL A,geo,1.0,1.0,1.0,21.081060,0.985946
5,Fisher Brewing Company,FISHER BREWING COMPANY - DOG PATIO,geo,1.0,1.0,1.0,21.199857,0.985867
6,Coffee Garden,COFFEE GARDEN,geo,1.0,1.0,1.0,22.386611,0.985076
7,Little Caesars Pizza,LITTLE CAESARS PIZZA,geo,1.0,1.0,1.0,23.137558,0.984575
8,Taqueria 27 Downtown,TAQUERIA 27 DOWNTOWN - DOG PATIO,geo,1.0,1.0,1.0,24.465891,0.983689
9,Subway,SUBWAY,geo,1.0,1.0,1.0,25.999576,0.982667


### Map check — matched pairs should land on the same location


In [9]:
# Geo matches: plot both endpoints. Text-fallback: meta pin only (no SLCHD lat/lon).
if match_audit.empty:
    print("No matches to plot.")
else:
    geo_rows = match_audit[match_audit["match_method"] == "geo"]
    text_rows = match_audit[match_audit["match_method"] == "text_fallback"]

    frames = []
    if not geo_rows.empty:
        frames.append(
            geo_rows.assign(
                side="meta (geo)",
                lat=geo_rows["meta_lat"],
                lon=geo_rows["meta_lon"],
                label=geo_rows["meta_name"],
            )
        )
        frames.append(
            geo_rows.assign(
                side="slchd (geo)",
                lat=geo_rows["slchd_lat"],
                lon=geo_rows["slchd_lon"],
                label=geo_rows["slchd_name"],
            )
        )
    if not text_rows.empty:
        frames.append(
            text_rows.assign(
                side="meta (text_fallback)",
                lat=text_rows["meta_lat"],
                lon=text_rows["meta_lon"],
                label=text_rows["meta_name"],
            )
        )

    map_df = pd.concat(frames, ignore_index=True).dropna(subset=["lat", "lon"])

    fig = px.scatter_map(
        map_df,
        lat="lat",
        lon="lon",
        color="side",
        hover_name="label",
        hover_data={
            "SLCHD_establishment_keys": True,
            "match_method": True,
            "dist_m": ":.1f",
            "score": ":.3f",
            "lat": False,
            "lon": False,
        },
        zoom=9,
        height=560,
        title=(
            f"Geo + text-fallback matches "
            f"(geo={len(geo_rows)}, text={len(text_rows)})"
        ),
    )
    fig.update_layout(map_style="open-street-map", margin=dict(l=0, r=0, t=40, b=0))
    fig.show()


### Write `meta` with `SLCHD_establishment_keys`


In [10]:
meta_matched = meta.merge(
    match_audit[["gmap_id", "SLCHD_establishment_keys"]],
    on="gmap_id",
    how="left",
)
# Keep SLCHD key as the rightmost column for easy scanning
cols = [c for c in meta_matched.columns if c != "SLCHD_establishment_keys"] + [
    "SLCHD_establishment_keys"
]
meta_matched = meta_matched[cols]
meta_matched.to_csv(PATHS["meta_matched"], index=False)

{
    "out": str(PATHS["meta_matched"]),
    "rows": len(meta_matched),
    "matched": int(meta_matched["SLCHD_establishment_keys"].notna().sum()),
    "unmatched": int(meta_matched["SLCHD_establishment_keys"].isna().sum()),
}


{'out': '/Users/arleth/Desktop/Research_Work/food-health-viz/data/final/meta-Utah_food_matched.csv',
 'rows': 8592,
 'matched': 139,
 'unmatched': 8453}

## Build final establishments tables

From `meta` (all Google Local places, not just SLCHD-matched) + the ER
result above:

- **`establishments`** — core fields per `gmap_id`. Keeps raw
  `latitude`/`longitude`; the DuckDB loader builds the `geometry` column
  (CSV has no native geometry type).
- **`establishment_attributes`** — `meta.MISC` flattened to one row per
  `(gmap_id, attribute)`. `MISC` is `{category: [values...]}`; the outer
  category label is dropped, only the leaf values are kept.
- **`establishment_categories`** — `meta.category_id_1..14` melted to one
  row per `(gmap_id, category_id)`.


In [11]:
establishments = meta_matched.rename(
    columns={"SLCHD_establishment_keys": "slchd_establishment_key"}
)[
    [
        "gmap_id",
        "name",
        "address",
        "description",
        "latitude",
        "longitude",
        "avg_rating",
        "num_of_reviews",
        "price",
        "slchd_establishment_key",
    ]
]
establishments.to_csv(PATHS["establishments"], index=False)

{
    "out": str(PATHS["establishments"]),
    "rows": len(establishments),
    "slchd_matched": int(establishments["slchd_establishment_key"].notna().sum()),
}


{'out': '/Users/arleth/Desktop/Research_Work/food-health-viz/data/final/establishments.csv',
 'rows': 8592,
 'slchd_matched': 139}

In [12]:
import json as _json


def misc_attributes(misc_raw) -> list[str]:
    """Flatten meta.MISC {"category": [values...]} into a flat, deduped attribute list."""
    if pd.isna(misc_raw):
        return []
    try:
        data = _json.loads(misc_raw)
    except (TypeError, ValueError):
        return []
    seen: list[str] = []
    for values in data.values():
        for v in values:
            if v not in seen:
                seen.append(v)
    return seen


establishment_attributes = pd.DataFrame(
    [
        {"gmap_id": gmap_id, "attribute": attr}
        for gmap_id, misc_raw in zip(meta["gmap_id"], meta["MISC"])
        for attr in misc_attributes(misc_raw)
    ]
).drop_duplicates().reset_index(drop=True)
establishment_attributes.to_csv(PATHS["establishment_attributes"], index=False)

{"out": str(PATHS["establishment_attributes"]), "rows": len(establishment_attributes)}


{'out': '/Users/arleth/Desktop/Research_Work/food-health-viz/data/final/establishment_attributes.csv',
 'rows': 163113}

In [13]:
cat_cols = [c for c in meta.columns if c.startswith("category_id_")]
establishment_categories = (
    meta[["gmap_id", *cat_cols]]
    .melt(id_vars="gmap_id", value_vars=cat_cols, value_name="category_id")
    .dropna(subset=["category_id"])
    .astype({"category_id": int})
    [["gmap_id", "category_id"]]
    .drop_duplicates()
    .sort_values(["gmap_id", "category_id"])
    .reset_index(drop=True)
)
establishment_categories.to_csv(PATHS["establishment_categories"], index=False)

{"out": str(PATHS["establishment_categories"]), "rows": len(establishment_categories)}


{'out': '/Users/arleth/Desktop/Research_Work/food-health-viz/data/final/establishment_categories.csv',
 'rows': 25799}

## Establishment opening hours

Build a day-level hours table from `meta.hours` for every place that has it —
keyed on `gmap_id` (not the SLCHD key), so it covers all of `meta`, not just
the ER-matched subset.

| column | meaning |
|---|---|
| `gmap_id` | Google Local place id |
| `day` | `0`=Sunday … `6`=Saturday |
| `open_min` | open time, ms since midnight |
| `open_max` | close time, ms since midnight |

Closed days are omitted. Overnight closes (e.g. 11AM–1AM) have `open_max` < `open_min`. Natural key is `(gmap_id, day)`.


In [14]:
import ast
import re

DAY_TO_IDX = {
    "Sunday": 0,
    "Monday": 1,
    "Tuesday": 2,
    "Wednesday": 3,
    "Thursday": 4,
    "Friday": 5,
    "Saturday": 6,
}
MS_PER_MIN = 60_000
MS_PER_DAY = 24 * 60 * MS_PER_MIN

_TIME_RE = re.compile(
    r"^\s*(\d{1,2})(?::(\d{2}))?\s*(AM|PM)?\s*$",
    re.IGNORECASE,
)


def _clock_to_ms(hour: int, minute: int, ampm: str | None) -> int:
    """Convert a wall-clock time to ms since midnight."""
    if ampm:
        ampm = ampm.upper()
        h = hour % 12
        if ampm == "PM":
            h += 12
        hour = h
    return (hour * 60 + minute) * MS_PER_MIN


def parse_clock_token(token: str, default_ampm: str | None = None) -> int | None:
    """Parse '11AM', '6:30PM', '12', '9:30' → ms since midnight."""
    m = _TIME_RE.match(token.strip())
    if not m:
        return None
    hour = int(m.group(1))
    minute = int(m.group(2) or 0)
    ampm = m.group(3) or default_ampm
    if ampm is None and hour > 12:
        # 24h-ish token without meridiem
        return (hour * 60 + minute) * MS_PER_MIN
    if ampm is None:
        return None  # need inherited AM/PM
    return _clock_to_ms(hour, minute, ampm)


def parse_open_span(span: str) -> tuple[int, int] | None:
    """Parse a Google hours span into (open_min, open_max) ms, or None if closed/unparseable."""
    s = span.replace("–", "-").replace("—", "-").strip()
    low = s.lower()
    if low == "closed":
        return None
    if "24 hour" in low or low in {"open 24 hours", "24 hours"}:
        return (0, MS_PER_DAY)

    if "-" not in s:
        return None
    left, right = s.split("-", 1)
    left, right = left.strip(), right.strip()

    # Inherit AM/PM: "12–5PM" → both PM; "7–10AM" → both AM
    right_m = _TIME_RE.match(right)
    left_m = _TIME_RE.match(left)
    if not right_m or not left_m:
        return None
    right_ampm = right_m.group(3)
    left_ampm = left_m.group(3) or right_ampm
    # If only left has meridiem (rare), push it to right
    right_ampm = right_ampm or left_ampm
    if not left_ampm or not right_ampm:
        return None

    open_min = parse_clock_token(left, left_ampm)
    open_max = parse_clock_token(right, right_ampm)
    if open_min is None or open_max is None:
        return None
    # Midnight labeled 12AM is 0; treat "12AM" close as end-of-day when open wasn't midnight
    # (Google uses 12AM for midnight end-of-span). Keep as 0 → overnight vs open_min.
    return (open_min, open_max)


def hours_json_to_rows(gmap_id: str, hours_raw) -> list[dict]:
    """Expand one meta.hours blob into per-day rows."""
    if pd.isna(hours_raw):
        return []
    try:
        data = ast.literal_eval(str(hours_raw))
    except (SyntaxError, ValueError):
        return []
    rows = []
    for day_name, span in data:
        day = DAY_TO_IDX.get(str(day_name))
        if day is None:
            continue
        parsed = parse_open_span(str(span))
        if parsed is None:
            continue
        open_min, open_max = parsed
        rows.append(
            {
                "gmap_id": gmap_id,
                "day": day,
                "open_min": open_min,
                "open_max": open_max,
            }
        )
    return rows


hour_rows: list[dict] = []
for _, r in meta.iterrows():
    hour_rows.extend(hours_json_to_rows(r["gmap_id"], r["hours"]))

gmap_hours = (
    pd.DataFrame(hour_rows, columns=["gmap_id", "day", "open_min", "open_max"])
    .astype({"day": "Int64", "open_min": "Int64", "open_max": "Int64"})
    .sort_values(["gmap_id", "day"])
    .reset_index(drop=True)
)
gmap_hours.to_csv(PATHS["gmap_hours"], index=False)

{
    "out": str(PATHS["gmap_hours"]),
    "rows": len(gmap_hours),
    "establishments": int(gmap_hours["gmap_id"].nunique()),
}


{'out': '/Users/arleth/Desktop/Research_Work/food-health-viz/data/final/gmap_hours.csv',
 'rows': 50744,
 'establishments': 7969}

In [15]:
# Sanity: sample a few establishments (day 0=Sun … 6=Sat)
gmap_hours.head(14)


,gmap_id,day,open_min,open_max
0,0x4065f3f9fd551245:0xf9bea78a9e4281,1,37800000,75600000
1,0x4065f3f9fd551245:0xf9bea78a9e4281,2,37800000,75600000
2,0x4065f3f9fd551245:0xf9bea78a9e4281,3,37800000,75600000
3,0x4065f3f9fd551245:0xf9bea78a9e4281,4,37800000,75600000
4,0x4065f3f9fd551245:0xf9bea78a9e4281,5,37800000,75600000
5,0x4065f3f9fd551245:0xf9bea78a9e4281,6,37800000,75600000
6,0x4065fce90a978907:0xeff8ea6da4a7e440,0,0,86400000
7,0x4065fce90a978907:0xeff8ea6da4a7e440,1,0,86400000
8,0x4065fce90a978907:0xeff8ea6da4a7e440,2,0,86400000
9,0x4065fce90a978907:0xeff8ea6da4a7e440,3,0,86400000


## Inspections, violations, violation codes

Normalize `merged_inspections` (one row per violation, inspection fields
repeated) into three tables:

- **`inspections`** — one row per inspection visit
  (`slchd_establishment_key`, `inspection_date`, `inspection_type_id`, `inspection_score`).
- **`violations`** — one row per violation actually cited. Inspections with
  zero violations get zero rows here (no placeholder row, unlike the flat CSV).
- **`violation_codes`** — one row per distinct raw `violation_code` string.
  Asterisks (`*`/`**`) are split into their own `asterisk_count` column;
  `code_text` keeps the base code. `critical` is a single value per code —
  see the earlier fix cell, which normalized the 3 codes that were
  inconsistently marked.

Known source-data limitation, not something this cleaning step tries to
resolve: 173 inspection groups (2.5%) have the same `violation_code`
repeated within one inspection (546 rows, 1.7% of all violations) — looks
like duplicate/OCR-artifact scraping of the same report (e.g. one copy has
`"c9mpsrtment"`, the other `"compartment"`) rather than two genuinely
separate citations. Left as-is rather than guessed apart.


In [16]:
# Same establishment_key formula as scripts/gen_establishment_keys.py / establishment_keys_final,
# so this joins cleanly onto establishment_keys_final.slchd_establishment_key.
insp = merged_inspections.copy()
insp["slchd_establishment_key"] = (
    insp["establishment_name"].str.strip()
    + "|"
    + insp["address"].str.strip()
    + "|"
    + insp["city_state_zip"].str.split(",", n=1).str[0].str.strip()
).str.upper()

INSP_KEY = ["slchd_establishment_key", "inspection_date", "inspection_type", "inspection_score"]


In [17]:
# "01 - Routine" -> id 1, description "Routine"
insp_type_strings = insp["inspection_type"].dropna().unique()
inspection_types = pd.DataFrame(
    {
        "inspection_type_id": [int(s.split(" - ", 1)[0]) for s in insp_type_strings],
        "description": [s.split(" - ", 1)[1] for s in insp_type_strings],
    }
).sort_values("inspection_type_id").reset_index(drop=True)
inspection_types.to_csv(PATHS["inspection_types"], index=False)
inspection_types


,inspection_type_id,description
0,1,Routine
1,2,Followup
2,3,Complaint Investigation
3,7,Critical Item


In [18]:
# Establishments with zero inspections on record get one placeholder row in
# merged_inspections with every inspection/violation field empty (per its README) —
# drop those here, they're not a real inspection visit.
inspections_key_map = (
    insp[insp["inspection_date"].notna()][INSP_KEY].drop_duplicates().reset_index(drop=True)
)
inspections_key_map.insert(0, "inspection_id", inspections_key_map.index + 1)

inspections = inspections_key_map.copy()
inspections["inspection_type_id"] = (
    inspections["inspection_type"]
    .map(lambda s: int(s.split(" - ", 1)[0]) if pd.notna(s) else pd.NA)
    .astype("Int64")
)
inspections = inspections[
    ["inspection_id", "slchd_establishment_key", "inspection_date", "inspection_type_id", "inspection_score"]
]
inspections.to_csv(PATHS["inspections"], index=False)

{"out": str(PATHS["inspections"]), "rows": len(inspections)}


{'out': '/Users/arleth/Desktop/Research_Work/food-health-viz/data/final/inspections.csv',
 'rows': 8020}

In [19]:
viol = insp[insp["violation_code"].notna() & (insp["violation_code"] != "")].copy()
viol["code_text"] = viol["violation_code"].str.rstrip("*")
viol["asterisk_count"] = viol["violation_code"].str.len() - viol["code_text"].str.len()

violation_codes = (
    viol[["violation_code", "code_text", "asterisk_count", "violation_critical"]]
    .drop_duplicates(subset=["violation_code"])
    .sort_values("violation_code")
    .reset_index(drop=True)
)
violation_codes.insert(0, "violation_code_id", violation_codes.index + 1)

# Join each violation to its inspection_id and violation_code_id. No NaNs in
# INSP_KEY for these rows (verified — every violation row has a complete key),
# so this is a safe many-to-one merge.
viol = viol.merge(inspections_key_map[["inspection_id", *INSP_KEY]], on=INSP_KEY, how="left")
viol = viol.merge(violation_codes[["violation_code", "violation_code_id"]], on="violation_code", how="left")

violations = viol[
    [
        "inspection_id",
        "violation_code_id",
        "violation_description",
        "violation_occurrences",
        "violation_cos",
        "violation_phr",
    ]
].reset_index(drop=True)
violations.insert(0, "violation_id", violations.index + 1)
violations.to_csv(PATHS["violations"], index=False)

violation_codes = violation_codes.drop(columns=["violation_code"]).rename(
    columns={"violation_critical": "critical"}
)
violation_codes.to_csv(PATHS["violation_codes"], index=False)

{
    "inspections": len(inspections),
    "violations": len(violations),
    "violation_codes": len(violation_codes),
    "violations_missing_inspection_id": int(viol["inspection_id"].isna().sum()),
}


{'inspections': 8020,
 'violations': 32029,
 'violation_codes': 237,
 'violations_missing_inspection_id': 0}